# Flight Delay Prediction — Clean Final Notebook

This notebook predicts whether a flight will be delayed using **flight schedule data** and **origin airport weather data**.

The notebook keeps output clean:

- summary tables instead of long prints
- graphs for class distribution, route delay rates, threshold tuning, ROC curve, precision-recall curve
- confusion-matrix graphs
- final Random Forest model trained on the **whole prepared dataset** after model evaluation

**Important:** A test split is still used first to measure performance honestly. After evaluation, the final model is retrained on the whole prepared dataset and saved.

In [ ]:
# Section 1 — Settings

FLIGHT_PATH = "/kaggle/input/datasets/kamalalqedra/bts-ontime-performance-2019-present-csv/bts_ontime_2019.csv"
WEATHER_PATH = "/kaggle/input/datasets/sehamhakimothman/asos-weather-data-2019-2026/historical_weather_data_2019_2026.csv"

DELAY_THRESHOLD_MINUTES = 15
WEATHER_TOLERANCE = "3h"
TEST_SPLIT_DATE = "2019-10-01"
MIN_ROUTE_COUNT = 100
RANDOM_STATE = 42

FINAL_MODEL_FILE = "final_random_forest_full_dataset.joblib"
FINAL_THRESHOLD_FILE = "final_random_forest_threshold.joblib"
FINAL_DATASET_FILE = "final_flight_weather_dataset.parquet"

In [ ]:
# Section 2 — Imports

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 30)

## 1. Helper functions

These functions keep the notebook cleaner and avoid repeating code.

In [ ]:
# Section 3 — Helper functions

def hhmm_to_minutes(x):
    """Convert HHMM values like 830 or 1435 into minutes after midnight."""
    if pd.isna(x):
        return np.nan
    try:
        x = int(float(x))
    except Exception:
        return np.nan
    hour = x // 100
    minute = x % 100
    if hour == 24:
        hour = 0
    if hour < 0 or hour > 23 or minute < 0 or minute > 59:
        return np.nan
    return hour * 60 + minute


def add_utc_time_by_timezone(df, local_col, tz_col, out_col):
    """Convert local airport datetime to timezone-naive UTC datetime."""
    df = df.copy()
    df[out_col] = pd.NaT

    for tz, idx in df.groupby(tz_col, dropna=True).groups.items():
        local_times = pd.to_datetime(df.loc[idx, local_col], errors="coerce")
        utc_times = (
            local_times
            .dt.tz_localize(tz, ambiguous="NaT", nonexistent="shift_forward")
            .dt.tz_convert("UTC")
            .dt.tz_localize(None)
        )
        df.loc[idx, out_col] = utc_times

    return df


def metric_table(y_true, y_pred, y_proba, model_name):
    """Create a compact metrics table."""
    return pd.DataFrame([{
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_delayed": precision_score(y_true, y_pred, zero_division=0),
        "recall_delayed": recall_score(y_true, y_pred, zero_division=0),
        "f1_delayed": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
    }])


def plot_confusion_matrix(y_true, y_pred, title):
    """Plot confusion matrix without printing raw arrays."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=["Not delayed", "Delayed"],
        values_format="d",
        ax=ax,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## 2. Load and clean the flight dataset

The target variable is:

`Delayed = 1` if `ArrDelayMinutes >= 15`, otherwise `Delayed = 0`.

In [ ]:
# Section 4 — Load and clean flight data

flight_cols = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek", "FlightDate",
    "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",
    "CRSDepTime", "CRSArrTime", "Cancelled", "Diverted",
    "CRSElapsedTime", "Distance", "ArrDelay", "ArrDelayMinutes",
    "DepDelay", "DepDelayMinutes"
]

flights = pd.read_csv(FLIGHT_PATH, usecols=flight_cols, low_memory=False)
flights.columns = flights.columns.str.strip()

flights = flights[(flights["Cancelled"] == 0) & (flights["Diverted"] == 0)].copy()
flights = flights.dropna(subset=["ArrDelayMinutes", "CRSDepTime", "CRSArrTime", "FlightDate"]).copy()

flights["Origin"] = flights["Origin"].astype(str).str.upper().str.strip()
flights["Dest"] = flights["Dest"].astype(str).str.upper().str.strip()
flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], errors="coerce")

flights["Delayed"] = (flights["ArrDelayMinutes"] >= DELAY_THRESHOLD_MINUTES).astype(int)

flights["dep_minutes"] = flights["CRSDepTime"].apply(hhmm_to_minutes)
flights["arr_minutes"] = flights["CRSArrTime"].apply(hhmm_to_minutes)
flights = flights.dropna(subset=["FlightDate", "dep_minutes", "arr_minutes"]).copy()

flights["sched_dep_local"] = flights["FlightDate"] + pd.to_timedelta(flights["dep_minutes"], unit="m")
flights["sched_arr_local"] = flights["FlightDate"] + pd.to_timedelta(flights["arr_minutes"], unit="m")

# If scheduled arrival is earlier than scheduled departure, arrival is on the next day.
overnight = flights["arr_minutes"] < flights["dep_minutes"]
flights.loc[overnight, "sched_arr_local"] += pd.Timedelta(days=1)

flight_summary = pd.DataFrame({
    "metric": ["clean_flight_rows", "unique_origins", "unique_destinations", "delay_rate"],
    "value": [
        len(flights),
        flights["Origin"].nunique(),
        flights["Dest"].nunique(),
        round(flights["Delayed"].mean(), 4),
    ]
})

display(flight_summary)

In [ ]:
# Section 5 — Target distribution graph

class_counts = flights["Delayed"].value_counts().sort_index()
class_labels = ["Not delayed", "Delayed"]

plt.figure(figsize=(6, 4))
plt.bar(class_labels, class_counts.values)
plt.title("Flight Delay Class Distribution")
plt.ylabel("Number of flights")
plt.tight_layout()
plt.show()

## 3. Load and clean the weather dataset

Weather observations are cleaned and restricted to the flight year.

In [ ]:
# Section 6 — Load and clean weather data

expected_weather_cols = [
    "station", "valid", "tmpf", "dwpf", "relh", "drct", "sknt", "p01i",
    "alti", "mslp", "vsby", "gust", "feel", "skyc1", "skyc2", "wxcodes"
]

weather_header = pd.read_csv(WEATHER_PATH, nrows=0).columns.tolist()
weather_usecols = [c for c in expected_weather_cols if c in weather_header]

weather = pd.read_csv(WEATHER_PATH, usecols=weather_usecols, low_memory=False)
weather.columns = weather.columns.str.strip()

for col in expected_weather_cols:
    if col not in weather.columns:
        weather[col] = np.nan

weather["station"] = weather["station"].astype(str).str.upper().str.strip()
weather["valid"] = pd.to_datetime(weather["valid"], errors="coerce", utc=True).dt.tz_localize(None)
weather = weather.dropna(subset=["station", "valid"]).copy()

weather = weather[(weather["valid"] >= "2019-01-01") & (weather["valid"] < "2020-01-02")].copy()
weather = weather.replace("M", np.nan)
weather["p01i"] = weather["p01i"].replace("T", 0.0)

numeric_weather_cols = ["tmpf", "dwpf", "relh", "drct", "sknt", "p01i", "alti", "mslp", "vsby", "gust", "feel"]
for col in numeric_weather_cols:
    weather[col] = pd.to_numeric(weather[col], errors="coerce")

flight_airports = set(flights["Origin"].unique()) | set(flights["Dest"].unique())
weather = weather[weather["station"].isin(flight_airports)].copy()

weather_summary = pd.DataFrame({
    "metric": ["clean_weather_rows", "weather_stations", "weather_start", "weather_end"],
    "value": [
        len(weather),
        weather["station"].nunique(),
        weather["valid"].min(),
        weather["valid"].max(),
    ]
})

display(weather_summary)

## 4. Match airport time zones

`CRSDepTime` is combined with `FlightDate` to create scheduled departure datetime. Then the local airport time is converted to UTC so it can be matched with weather time.

In [ ]:
# Section 7 — Airport time zone mapping

origin_airports = flights[["Origin", "OriginState"]].rename(columns={"Origin": "airport", "OriginState": "state"})
dest_airports = flights[["Dest", "DestState"]].rename(columns={"Dest": "airport", "DestState": "state"})

airport_states = pd.concat([origin_airports, dest_airports], ignore_index=True)
airport_states = airport_states.dropna().drop_duplicates()
airport_states = (
    airport_states
    .groupby("airport")["state"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

state_to_tz = {
    "CT": "America/New_York", "DE": "America/New_York", "DC": "America/New_York",
    "FL": "America/New_York", "GA": "America/New_York", "ME": "America/New_York",
    "MD": "America/New_York", "MA": "America/New_York", "NH": "America/New_York",
    "NJ": "America/New_York", "NY": "America/New_York", "NC": "America/New_York",
    "OH": "America/New_York", "PA": "America/New_York", "RI": "America/New_York",
    "SC": "America/New_York", "VT": "America/New_York", "VA": "America/New_York",
    "WV": "America/New_York", "MI": "America/New_York", "KY": "America/New_York",
    "IN": "America/Indiana/Indianapolis",
    "AL": "America/Chicago", "AR": "America/Chicago", "IL": "America/Chicago",
    "IA": "America/Chicago", "LA": "America/Chicago", "MN": "America/Chicago",
    "MS": "America/Chicago", "MO": "America/Chicago", "OK": "America/Chicago",
    "WI": "America/Chicago", "TX": "America/Chicago", "SD": "America/Chicago",
    "ND": "America/Chicago", "NE": "America/Chicago", "KS": "America/Chicago",
    "TN": "America/Chicago",
    "AZ": "America/Phoenix", "CO": "America/Denver", "MT": "America/Denver",
    "NM": "America/Denver", "UT": "America/Denver", "WY": "America/Denver",
    "ID": "America/Denver",
    "CA": "America/Los_Angeles", "NV": "America/Los_Angeles", "WA": "America/Los_Angeles",
    "OR": "America/Los_Angeles",
    "AK": "America/Anchorage", "HI": "Pacific/Honolulu",
    "PR": "America/Puerto_Rico", "VI": "America/St_Thomas",
    "GU": "Pacific/Guam", "AS": "Pacific/Pago_Pago",
}

airport_tz_overrides = {
    "MIA": "America/New_York", "FLL": "America/New_York", "PBI": "America/New_York",
    "MCO": "America/New_York", "TPA": "America/New_York", "RSW": "America/New_York",
    "JAX": "America/New_York", "TLH": "America/New_York", "EYW": "America/New_York",
    "PNS": "America/Chicago", "VPS": "America/Chicago", "ECP": "America/Chicago",
    "DFW": "America/Chicago", "DAL": "America/Chicago", "IAH": "America/Chicago",
    "HOU": "America/Chicago", "AUS": "America/Chicago", "SAT": "America/Chicago",
    "ELP": "America/Denver",
    "BOI": "America/Denver", "IDA": "America/Denver", "SUN": "America/Denver",
    "GEG": "America/Los_Angeles",
    "PDX": "America/Los_Angeles", "EUG": "America/Los_Angeles",
    "MFR": "America/Los_Angeles", "OTH": "America/Los_Angeles",
    "FAR": "America/Chicago", "BIS": "America/Chicago", "MOT": "America/Chicago",
    "RAP": "America/Denver", "FSD": "America/Chicago", "OMA": "America/Chicago",
    "LNK": "America/Chicago", "MHK": "America/Chicago", "ICT": "America/Chicago",
    "SDF": "America/New_York", "LEX": "America/New_York", "BNA": "America/Chicago",
    "MEM": "America/Chicago", "TYS": "America/New_York", "CHA": "America/New_York",
    "IND": "America/Indiana/Indianapolis", "FWA": "America/Indiana/Indianapolis",
    "SBN": "America/Indiana/Indianapolis", "DTW": "America/New_York",
    "GRR": "America/New_York", "TVC": "America/New_York", "MQT": "America/New_York",
    "ABI": "America/Chicago", "ACT": "America/Chicago", "AMA": "America/Chicago",
    "BPT": "America/Chicago", "BRO": "America/Chicago", "CLL": "America/Chicago",
    "CRP": "America/Chicago", "DRT": "America/Chicago", "GGG": "America/Chicago",
    "GRK": "America/Chicago", "HRL": "America/Chicago", "LBB": "America/Chicago",
    "LRD": "America/Chicago", "MAF": "America/Chicago", "MFE": "America/Chicago",
    "SJT": "America/Chicago", "SPS": "America/Chicago", "TYR": "America/Chicago",
    "ABR": "America/Chicago", "ATY": "America/Chicago", "PIR": "America/Chicago",
    "DVL": "America/Chicago", "GFK": "America/Chicago", "ISN": "America/Chicago",
    "JMS": "America/Chicago", "XWA": "America/Chicago", "BFF": "America/Denver",
    "EAR": "America/Chicago", "GRI": "America/Chicago", "LBF": "America/Chicago",
    "GCK": "America/Chicago", "HYS": "America/Chicago", "LBL": "America/Chicago",
    "SLN": "America/Chicago", "APN": "America/New_York", "AZO": "America/New_York",
    "CIU": "America/New_York", "CMX": "America/New_York", "ESC": "America/New_York",
    "FNT": "America/New_York", "IMT": "America/New_York", "LAN": "America/New_York",
    "MBS": "America/New_York", "MKG": "America/New_York", "PLN": "America/New_York",
    "EVV": "America/Chicago", "CVG": "America/New_York", "OWB": "America/Chicago",
    "PAH": "America/Chicago", "TRI": "America/New_York", "PIH": "America/Denver",
    "TWF": "America/Denver", "LWS": "America/Los_Angeles", "RDM": "America/Los_Angeles",
    "DAB": "America/New_York", "GNV": "America/New_York", "MLB": "America/New_York",
    "PGD": "America/New_York", "PIE": "America/New_York", "SFB": "America/New_York",
    "SRQ": "America/New_York", "GUM": "Pacific/Guam", "SPN": "Pacific/Saipan",
    "PPG": "Pacific/Pago_Pago",
}

airport_states["timezone"] = airport_states["state"].map(state_to_tz)
airport_states["timezone"] = airport_states.apply(
    lambda row: airport_tz_overrides.get(row["airport"], row["timezone"]),
    axis=1,
)

flights = flights.merge(
    airport_states[["airport", "timezone"]].rename(columns={"airport": "Origin", "timezone": "OriginTimezone"}),
    on="Origin",
    how="left",
)

flights = add_utc_time_by_timezone(
    flights,
    local_col="sched_dep_local",
    tz_col="OriginTimezone",
    out_col="sched_dep_utc",
)

timezone_summary = pd.DataFrame({
    "metric": ["missing_origin_timezone_rows", "missing_sched_dep_utc_rows"],
    "value": [flights["OriginTimezone"].isna().sum(), flights["sched_dep_utc"].isna().sum()]
})

display(timezone_summary)

## 5. Join flights with origin weather

The join uses:

- `Origin` from flight data = `station` from weather data
- `sched_dep_utc` from flight data matched to the latest `valid_utc` weather observation before departure

This uses scheduled CRS departure time correctly because `CRSDepTime` was first converted into a real datetime using `FlightDate + CRSDepTime`.

In [ ]:
# Section 8 — Prepare weather table for joining

weather_join = weather[[
    "station", "valid", "tmpf", "dwpf", "relh", "drct", "sknt", "p01i",
    "alti", "mslp", "vsby", "feel", "skyc1", "skyc2", "wxcodes"
]].copy()

weather_join = weather_join.rename(columns={"valid": "valid_utc"})
weather_join = weather_join.dropna(subset=["station", "valid_utc"]).copy()
weather_join = weather_join.drop_duplicates(subset=["station", "valid_utc"]).copy()
weather_join = weather_join.sort_values(["station", "valid_utc"]).reset_index(drop=True)

weather_airports = set(weather_join["station"].dropna().unique())
flights_wx_origin = flights[
    flights["Origin"].isin(weather_airports) & flights["sched_dep_utc"].notna()
].copy()

join_summary = pd.DataFrame({
    "metric": ["flight_rows_before_weather_filter", "flight_rows_with_origin_weather", "weather_stations_used"],
    "value": [len(flights), len(flights_wx_origin), len(weather_airports)]
})

display(join_summary)

In [ ]:
# Section 9 — Merge latest origin weather before departure

def merge_origin_weather(flights_df, weather_df, tolerance=WEATHER_TOLERANCE):
    parts = []
    tolerance = pd.Timedelta(tolerance)

    weather_cols_to_prefix = [
        col for col in weather_df.columns
        if col not in ["station", "valid_utc"]
    ]

    for airport, f_part in flights_df.groupby("Origin", sort=False):
        w_part = weather_df[weather_df["station"] == airport].copy()
        if w_part.empty:
            continue

        merged = pd.merge_asof(
            f_part.sort_values("sched_dep_utc"),
            w_part.sort_values("valid_utc"),
            left_on="sched_dep_utc",
            right_on="valid_utc",
            direction="backward",
            tolerance=tolerance,
        )
        parts.append(merged)

    if not parts:
        raise ValueError("No flight-weather matches were found. Check airport codes and time columns.")

    out = pd.concat(parts, ignore_index=True)
    out = out.rename(columns={col: f"origin_wx_{col}" for col in weather_cols_to_prefix})
    out = out.rename(columns={"valid_utc": "origin_wx_valid_utc"})
    return out

model_data = merge_origin_weather(flights_wx_origin, weather_join)
model_data["origin_wx_age_min"] = (
    model_data["sched_dep_utc"] - model_data["origin_wx_valid_utc"]
).dt.total_seconds() / 60

model_data = model_data[model_data["origin_wx_valid_utc"].notna()].copy()

merge_summary = pd.DataFrame({
    "metric": ["rows_after_weather_merge", "delay_rate_after_merge", "median_weather_age_minutes"],
    "value": [
        len(model_data),
        round(model_data["Delayed"].mean(), 4),
        round(model_data["origin_wx_age_min"].median(), 2),
    ]
})

display(merge_summary)

## 6. Feature engineering

Features are created only from information available before the flight.

In [ ]:
# Section 10 — Feature engineering

model_data["dep_hour"] = model_data["sched_dep_local"].dt.hour
model_data["dep_month"] = model_data["sched_dep_local"].dt.month
model_data["dep_dayofweek"] = model_data["sched_dep_local"].dt.dayofweek
model_data["is_weekend"] = model_data["dep_dayofweek"].isin([5, 6]).astype(int)

model_data["route"] = model_data["Origin"] + "_" + model_data["Dest"]

wx = model_data["origin_wx_wxcodes"].fillna("").astype(str)
model_data["wx_rain"] = wx.str.contains("RA", regex=False).astype(int)
model_data["wx_snow"] = wx.str.contains("SN", regex=False).astype(int)
model_data["wx_fog"] = wx.str.contains("FG|BR", regex=True).astype(int)
model_data["wx_thunder"] = wx.str.contains("TS", regex=False).astype(int)

feature_summary = pd.DataFrame({
    "metric": ["final_model_rows", "unique_routes", "unique_airlines", "final_delay_rate"],
    "value": [
        len(model_data),
        model_data["route"].nunique(),
        model_data["Reporting_Airline"].nunique(),
        round(model_data["Delayed"].mean(), 4),
    ]
})

display(feature_summary)

In [ ]:
# Section 11 — Route delay-rate graph for common routes

route_summary = (
    model_data
    .groupby("route")
    .agg(total_flights=("Delayed", "size"), delay_rate=("Delayed", "mean"))
    .query("total_flights >= @MIN_ROUTE_COUNT")
    .sort_values("delay_rate", ascending=False)
)

top_routes = route_summary.head(15).sort_values("delay_rate")

plt.figure(figsize=(9, 6))
plt.barh(top_routes.index, top_routes["delay_rate"])
plt.xlabel("Delay rate")
plt.ylabel("Route")
plt.title("Top 15 High-Delay Routes with Enough Flights")
plt.tight_layout()
plt.show()

## 7. Evaluation split

This split is used only to estimate performance honestly. After evaluation, the final model is trained on the whole prepared dataset.

In [ ]:
# Section 12 — Train/test split for honest evaluation

split_date = pd.Timestamp(TEST_SPLIT_DATE)
train_df = model_data[model_data["FlightDate"] < split_date].copy()
test_df = model_data[model_data["FlightDate"] >= split_date].copy()

if len(train_df) == 0 or len(test_df) == 0:
    train_df, test_df = train_test_split(
        model_data,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=model_data["Delayed"],
    )

# Rare routes are grouped using training data only to avoid leakage.
common_routes = train_df["route"].value_counts()
common_routes = common_routes[common_routes >= MIN_ROUTE_COUNT].index

train_df["route_clean"] = train_df["route"].where(train_df["route"].isin(common_routes), "OTHER")
test_df["route_clean"] = test_df["route"].where(test_df["route"].isin(common_routes), "OTHER")

split_summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "delay_rate": [round(train_df["Delayed"].mean(), 4), round(test_df["Delayed"].mean(), 4)]
})

display(split_summary)

In [ ]:
# Section 13 — Define features

numeric_features = [
    "dep_hour", "dep_month", "dep_dayofweek", "is_weekend",
    "CRSElapsedTime", "Distance",
    "origin_wx_tmpf", "origin_wx_dwpf", "origin_wx_relh",
    "origin_wx_drct", "origin_wx_sknt", "origin_wx_p01i",
    "origin_wx_alti", "origin_wx_mslp", "origin_wx_vsby",
    "origin_wx_feel", "origin_wx_age_min",
    "wx_rain", "wx_snow", "wx_fog", "wx_thunder",
]

categorical_features = [
    "Reporting_Airline", "Origin", "Dest", "route_clean",
    "origin_wx_skyc1", "origin_wx_skyc2",
]

X_train = train_df[numeric_features + categorical_features]
y_train = train_df["Delayed"]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df["Delayed"]

## 8. Train and evaluate Random Forest

Random Forest is used because it performed best in the previous notebook compared with Logistic Regression and Neural Network.

In [ ]:
# Section 14 — Train Random Forest for evaluation

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

rf_eval_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

rf_eval_model.fit(X_train, y_train)

rf_test_proba = rf_eval_model.predict_proba(X_test)[:, 1]
rf_test_pred_050 = (rf_test_proba >= 0.50).astype(int)

eval_metrics_050 = metric_table(y_test, rf_test_pred_050, rf_test_proba, "Random Forest, threshold 0.50")
display(eval_metrics_050.round(4))

In [ ]:
# Section 15 — Classification report table

report_050 = pd.DataFrame(
    classification_report(
        y_test,
        rf_test_pred_050,
        target_names=["Not delayed", "Delayed"],
        output_dict=True,
        zero_division=0,
    )
).T

display(report_050.round(3))

In [ ]:
# Section 16 — Confusion matrix graph at threshold 0.50

plot_confusion_matrix(
    y_test,
    rf_test_pred_050,
    "Random Forest Confusion Matrix — Test Set, Threshold 0.50"
)

In [ ]:
# Section 17 — ROC curve and Precision-Recall curve

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, rf_test_proba, ax=ax)
ax.set_title("Random Forest ROC Curve — Test Set")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(y_test, rf_test_proba, ax=ax)
ax.set_title("Random Forest Precision-Recall Curve — Test Set")
plt.tight_layout()
plt.show()

## 9. Threshold tuning

For delay detection, the default threshold `0.50` may miss too many delayed flights. This section chooses the threshold with the best delayed-class F1-score.

In [ ]:
# Section 18 — Threshold tuning table

threshold_results = []

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    pred_t = (rf_test_proba >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, pred_t),
        "precision_delayed": precision_score(y_test, pred_t, zero_division=0),
        "recall_delayed": recall_score(y_test, pred_t, zero_division=0),
        "f1_delayed": f1_score(y_test, pred_t, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results)
best_threshold = threshold_df.sort_values("f1_delayed", ascending=False).iloc[0]["threshold"]

display(threshold_df.round(4))

In [ ]:
# Section 19 — Threshold tuning graph

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision_delayed"], marker="o", label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall_delayed"], marker="o", label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1_delayed"], marker="o", label="F1-score")
plt.axvline(best_threshold, linestyle="--", label=f"Best threshold = {best_threshold:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning for Delayed Flights")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Section 20 — Confusion matrix graph using best threshold

rf_test_pred_best = (rf_test_proba >= best_threshold).astype(int)

eval_metrics_best = metric_table(
    y_test,
    rf_test_pred_best,
    rf_test_proba,
    f"Random Forest, threshold {best_threshold:.2f}"
)

display(eval_metrics_best.round(4))

plot_confusion_matrix(
    y_test,
    rf_test_pred_best,
    f"Random Forest Confusion Matrix — Test Set, Threshold {best_threshold:.2f}"
)

## 10. Train final model on the whole prepared dataset

After evaluation, the final model is trained on the whole prepared dataset. The confusion matrix in this section is **training performance only**, not a fair estimate of future performance.

In [ ]:
# Section 21 — Prepare full dataset for final training

full_model_data = model_data.copy()

full_common_routes = full_model_data["route"].value_counts()
full_common_routes = full_common_routes[full_common_routes >= MIN_ROUTE_COUNT].index
full_model_data["route_clean"] = full_model_data["route"].where(full_model_data["route"].isin(full_common_routes), "OTHER")

final_X = full_model_data[numeric_features + categorical_features]
final_y = full_model_data["Delayed"]

full_training_summary = pd.DataFrame({
    "metric": ["final_training_rows", "final_training_features", "final_delay_rate", "final_threshold"],
    "value": [len(final_X), final_X.shape[1], round(final_y.mean(), 4), best_threshold]
})

display(full_training_summary)

In [ ]:
# Section 22 — Train final Random Forest on the whole prepared dataset

final_random_forest_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

final_random_forest_model.fit(final_X, final_y)

final_train_proba = final_random_forest_model.predict_proba(final_X)[:, 1]
final_train_pred = (final_train_proba >= best_threshold).astype(int)

final_training_metrics = metric_table(
    final_y,
    final_train_pred,
    final_train_proba,
    "Final Random Forest trained on whole dataset"
)

display(final_training_metrics.round(4))

In [ ]:
# Section 23 — Full-dataset training confusion matrix graph

plot_confusion_matrix(
    final_y,
    final_train_pred,
    "Final Model Confusion Matrix — Whole Dataset Training Check"
)

## 11. Save final artifacts

The saved model can be reused later for prediction. The saved dataset is the cleaned, joined flight-weather dataset.

In [ ]:
# Section 24 — Save model, threshold, and final dataset

joblib.dump(final_random_forest_model, FINAL_MODEL_FILE)
joblib.dump(best_threshold, FINAL_THRESHOLD_FILE)
full_model_data.to_parquet(FINAL_DATASET_FILE, index=False)

saved_files = pd.DataFrame({
    "file": [FINAL_MODEL_FILE, FINAL_THRESHOLD_FILE, FINAL_DATASET_FILE],
    "exists": [os.path.exists(FINAL_MODEL_FILE), os.path.exists(FINAL_THRESHOLD_FILE), os.path.exists(FINAL_DATASET_FILE)],
    "size_mb": [
        round(os.path.getsize(FINAL_MODEL_FILE) / (1024 ** 2), 2),
        round(os.path.getsize(FINAL_THRESHOLD_FILE) / (1024 ** 2), 4),
        round(os.path.getsize(FINAL_DATASET_FILE) / (1024 ** 2), 2),
    ],
})

display(saved_files)

In [ ]:
# Section 25 — Test loading the saved model

loaded_model = joblib.load(FINAL_MODEL_FILE)
loaded_threshold = joblib.load(FINAL_THRESHOLD_FILE)

sample_proba = loaded_model.predict_proba(final_X.head(10))[:, 1]
sample_pred = (sample_proba >= loaded_threshold).astype(int)

sample_predictions = final_X.head(10).copy()
sample_predictions["delay_probability"] = sample_proba
sample_predictions["predicted_delayed"] = sample_pred

display(sample_predictions[["delay_probability", "predicted_delayed"]].round(4))

## Final conclusion

This notebook created a clean flight delay prediction pipeline. Flight data was joined with weather data using origin airport and scheduled departure time. The Random Forest model was evaluated on a test set, the best threshold was selected using delayed-class F1-score, and the final model was retrained on the whole prepared dataset.

Use the **test-set metrics** for reporting performance. The whole-dataset metrics are only a training check after the final model has been retrained on all available prepared data.